# Step 1 — Extract Dataset from JPEG Images

Reads camera frames from a folder of JPEG images. No ROS installation needed.

**Output files (saved to `data/`):**
- `X_features.npy` — feature matrix, shape (N, 4098) — flattened 64×64 edge patch + x_norm + angle_norm
- `y_labels.npy` — integer class labels (0=LEFT, 1=STRAIGHT, 2=RIGHT)

## Feature pipeline (Lab07b)

$$
\text{BGR frame} \rightarrow \text{resize 320×180} \rightarrow \text{grayscale} \rightarrow \text{ROI (bottom 55\%)} \rightarrow \text{Canny edges} \rightarrow \text{Hough segments} \rightarrow \text{64×64 patch} + \text{x\_norm} + \text{angle\_norm} \rightarrow \text{flatten}
$$

## Label scheme (image-based)

Labels are derived from the **position of the detected line in the image**, not from what the driver did.
The segment x-centre (`x_norm`) tells us where the line is relative to the camera:

| x_norm | Meaning | Required steering |
|--------|---------|------------------|
| < −0.10 | Line is left of centre | Steer LEFT to follow it |
| −0.10 … +0.10 | Line is centred | STRAIGHT |
| > +0.10 | Line is right of centre | Steer RIGHT to follow it |

This is independent of how the driver actually steered and gives clean, geometry-based labels.

## 1. Imports and paths

**Set `IMAGES_DIR` to the folder containing your 48,000 JPEG files.**

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

REPO_ROOT  = Path("D:/HSHL_resources_v2/6th_Semester/Autonomous_Systems_A/Autonomous_Systems_A_Lab_Group_4")

# ─── CHANGE THIS to your images folder ────────────────────────────────────────
IMAGES_DIR = REPO_ROOT / "data" / "extracted_images"
# ──────────────────────────────────────────────────────────────────────────────

DATA_DIR   = REPO_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

CLASS_NAMES = np.array(["LEFT", "STRAIGHT", "RIGHT"])

# Collect all JPEG/JPG files
image_paths = sorted(
    list(IMAGES_DIR.glob("*.jpg")) + list(IMAGES_DIR.glob("*.jpeg")) + list(IMAGES_DIR.glob("*.JPG"))
)

assert len(image_paths) > 0, f"No JPEG files found in:\n{IMAGES_DIR}"
print(f"Images directory: {IMAGES_DIR}")
print(f"Total images found: {len(image_paths)}")
print(f"First image: {image_paths[0].name}")
print(f"Last  image: {image_paths[-1].name}")

## 2. Feature pipeline (Lab07b)

In [ ]:
PATCH_SIZE   = 64
TARGET_W     = 320
TARGET_H     = 180
ROI_FRACTION = 0.45   # keep bottom 55%


def compute_edges(roi_gray: np.ndarray) -> np.ndarray:
    blurred = cv2.GaussianBlur(roi_gray, (5, 5), 0)
    return cv2.Canny(blurred, 50, 150)


def dominant_segment(edges: np.ndarray):
    """Return (x_center, y_center, angle_deg) of the longest Hough segment, or None."""
    lines = cv2.HoughLinesP(
        edges,
        rho=1, theta=np.pi / 180,
        threshold=25,
        minLineLength=25,
        maxLineGap=15,
    )
    if lines is None:
        return None
    best, best_len, best_angle = None, 0, 0.0
    for x1, y1, x2, y2 in lines[:, 0]:
        length = np.hypot(x2 - x1, y2 - y1)
        if length > best_len:
            best_len = length
            best = ((x1 + x2) / 2, (y1 + y2) / 2)
            best_angle = float(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
    return best[0], best[1], best_angle


def crop_patch(edge_img: np.ndarray, cx: float, cy: float, size: int = PATCH_SIZE) -> np.ndarray:
    h, w = edge_img.shape
    half = size // 2
    x1 = max(0, int(cx) - half);  x2 = min(w, int(cx) + half)
    y1 = max(0, int(cy) - half);  y2 = min(h, int(cy) + half)
    patch = edge_img[y1:y2, x1:x2]
    return cv2.resize(patch, (size, size))


def extract_features(img_bgr: np.ndarray):
    """
    Returns (feature_vector, x_norm) where feature_vector is float32 shape (4098,):
      - 4096 values: flattened 64x64 edge patch
      -    1 value:  x_norm — segment x-centre in [-1, 1] (left=-1, right=+1)
      -    1 value:  angle_norm — segment angle / 90
    Returns (None, None) if no segment found.
    """
    img   = cv2.resize(img_bgr, (TARGET_W, TARGET_H))
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    y0    = int(TARGET_H * ROI_FRACTION)
    roi   = gray[y0:, :]
    edges = compute_edges(roi)
    seg   = dominant_segment(edges)
    if seg is None:
        return None, None
    cx, cy, angle = seg
    patch      = crop_patch(edges, cx, cy)
    patch_feat = (patch.astype(np.float32) / 255.0).flatten()
    x_norm     = np.float32((cx / roi.shape[1]) * 2.0 - 1.0)
    a_norm     = np.float32(angle / 90.0)
    return np.append(patch_feat, [x_norm, a_norm]), float(x_norm)


print("Feature helpers defined (Lab07b pipeline).")

## 3. Label scheme — image-based

The label is derived from `x_norm` (where the line is in the frame), not from the driver's steering.

- Line left of centre (`x_norm < -DEAD_BAND`) → car must steer **LEFT** to follow it
- Line right of centre (`x_norm > +DEAD_BAND`) → car must steer **RIGHT** to follow it
- Line centred → **STRAIGHT**

In [ ]:
DEAD_BAND = 0.10   # x_norm threshold — tune if class distribution is too skewed

def xnorm_to_label(x_norm: float) -> int:
    if x_norm < -DEAD_BAND:
        return 0  # LEFT
    elif x_norm > DEAD_BAND:
        return 2  # RIGHT
    return 1      # STRAIGHT

print(f"Dead-band: ±{DEAD_BAND}")
print("x_norm < -0.10  →  LEFT")
print("-0.10 to +0.10  →  STRAIGHT")
print("x_norm > +0.10  →  RIGHT")

## 4. Read images and compute features

Every image in `IMAGES_DIR` is loaded, features are extracted, and the label is assigned from `x_norm`.

In [ ]:
SUBSAMPLE = 1   # 1 = use every image; 2 = every other image, etc.

X_list, x_norms = [], []
sample_img = None
skipped = 0

paths_to_use = image_paths[::SUBSAMPLE]
print(f"Processing {len(paths_to_use)} images (SUBSAMPLE={SUBSAMPLE})...")

for idx, img_path in enumerate(paths_to_use):
    img = cv2.imread(str(img_path))
    if img is None:
        skipped += 1
        continue

    feat, x_norm = extract_features(img)
    if feat is None:
        skipped += 1
        continue

    X_list.append(feat)
    x_norms.append(x_norm)

    if sample_img is None:
        sample_img = img.copy()

    if (idx + 1) % 5000 == 0:
        print(f"  {idx + 1}/{len(paths_to_use)} images processed  ({len(X_list)} features extracted)")

print(f"\nDone. Extracted: {len(X_list)} features  |  skipped (no segment / unreadable): {skipped}")

if sample_img is not None:
    plt.figure(figsize=(8, 4))
    plt.imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
    plt.title("Sample frame")
    plt.axis("off")
    plt.show()

## 5. Build labels from x_norm and inspect distribution

In [ ]:
X        = np.array(X_list,  dtype=np.float32)
x_norms  = np.array(x_norms, dtype=np.float32)
y_labels = np.array([xnorm_to_label(x) for x in x_norms], dtype=np.int32)

print(f"Feature matrix X: {X.shape}")
print(f"Labels y:         {y_labels.shape}")
print()
print("Class distribution:")
for i, name in enumerate(CLASS_NAMES):
    mask = y_labels == i
    n    = mask.sum()
    mean_x = x_norms[mask].mean() if n > 0 else 0.0
    print(f"  {name:10s}: {n:5d}  ({100*n/len(y_labels):.1f}%)  mean x_norm={mean_x:+.3f}")

# Plot x_norm distribution with class boundaries
plt.figure(figsize=(8, 3))
plt.hist(x_norms, bins=80, color="steelblue", edgecolor="none")
plt.axvline(-DEAD_BAND, color="red",   linestyle="--", linewidth=1.5, label=f"±{DEAD_BAND} dead-band")
plt.axvline( DEAD_BAND, color="red",   linestyle="--", linewidth=1.5)
plt.xlabel("x_norm (line position: −1=left edge, 0=centre, +1=right edge)")
plt.ylabel("Frame count")
plt.title("Line position distribution — image-based labels")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Sanity check — mean x_norm per class

Because labels come directly from x_norm, the mean x_norm per class should be clearly separated:
- LEFT mean x_norm should be negative
- RIGHT mean x_norm should be positive
- STRAIGHT should be near zero

If this is not the case, adjust `DEAD_BAND` in cell 3.

In [ ]:
print("Mean x_norm per class (should be clearly separated):")
for i, name in enumerate(CLASS_NAMES):
    mask = y_labels == i
    vals = x_norms[mask]
    print(f"  {name:10s}: mean={vals.mean():+.3f}  min={vals.min():+.3f}  max={vals.max():+.3f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
colors = ["#e74c3c", "#2ecc71", "#3498db"]
for i, (name, color) in enumerate(zip(CLASS_NAMES, colors)):
    axes[i].hist(x_norms[y_labels == i], bins=40, color=color)
    axes[i].set_title(f"{name} (n={int((y_labels==i).sum())})")
    axes[i].set_xlabel("x_norm")
axes[0].set_ylabel("Count")
plt.suptitle("x_norm distribution per class")
plt.tight_layout()
plt.show()

## 7. Save to data/

In [ ]:
np.save(str(DATA_DIR / "X_features.npy"), X)
np.save(str(DATA_DIR / "y_labels.npy"),   y_labels)

print("Saved:")
for f in sorted(DATA_DIR.glob("*.npy")):
    print(f"  {f.name}  ({f.stat().st_size // 1024} KB)")
print("\nDone. Run 02_train_svm.ipynb next.")